In [1]:
from pathlib import Path

import polars as pl
from polars import selectors as cs

pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_float("full")


polars.config.Config

In [2]:
data_assemblies = pl.concat(
    pl.read_csv(
        p / "quast_assembly.csv",
        infer_schema_length=None,
        null_values=[""],
    )
    for p in sorted(Path("../data").iterdir())
    if p.is_dir()
)

In [3]:
data_bins = pl.concat(
    [
        pl.read_csv(
            p / "bin_summary.tsv",
            separator="\t",
            infer_schema_length=None,
            null_values=[""],
        )
        .with_columns(dataset=pl.lit(p.name))
        .drop(cs.starts_with("Depth"))
        for p in sorted(Path("../data").iterdir())
        if p.is_dir()
    ],
    how="diagonal_relaxed",
).with_columns(assembler=pl.col("bin").str.split("-").list.get(0))

In [4]:
# Summary of the assemblies per dataset and assembler
data_assemblies.group_by(["dataset", "assembler"]).agg(
    mean_length=pl.col("Total length").mean().round(0),
    mean_n_contigs=pl.col("# contigs (>= 0 bp)").mean().round(0),
    mean_n50=pl.col("N50").mean().round(0),
).sort(["dataset", "assembler"])

dataset,assembler,mean_length,mean_n_contigs,mean_n50
str,str,f64,f64,f64
"""maghini""","""FLYE""",301556887,5657,202598
"""maghini""","""MEGAHIT""",375361696,238407,11842
"""maghini""","""METAMDBG""",328894787,7042,177966
"""maghini""","""SPAdes""",375211098,466571,11970
"""maghini""","""SPAdesHybrid""",414451729,386373,34351
"""zymo""","""FLYE""",553848891,12178,104289
"""zymo""","""MEGAHIT""",549466815,544106,3990
"""zymo""","""METAMDBG""",669964415,14683,134635
"""zymo""","""SPAdes""",505226327,1304494,3204


In [5]:
# Summary of the bins per dataset and assembler
data_bins.group_by(["dataset", "assembler"]).agg(
    number_of_bins=pl.len(), mean_length=pl.col("Total length_quast").mean().round(0)
).sort(["dataset", "assembler"])

dataset,assembler,number_of_bins,mean_length
str,str,u32,f64
"""maghini""","""FLYE""",8207,2172800
"""maghini""","""MEGAHIT""",8202,2188133
"""maghini""","""METAMDBG""",9373,2041121
"""maghini""","""SPAdes""",8180,2212225
"""maghini""","""SPAdesHybrid""",9855,2186117
"""zymo""","""FLYE""",1644,1876010
"""zymo""","""MEGAHIT""",1074,2253738
"""zymo""","""METAMDBG""",1960,1941960
"""zymo""","""SPAdes""",1058,2055474
